<div dir="rtl">

# 2. پاکسازی و آماد هسازی دادهها و تحلیل اکتشافی

</div>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
df = pd.read_csv('./covid_19.csv')

df_clean = df[df['icu'].isin([1, 2])].copy()

# # برای راحتی کار، مقادیر 1 و 2 را به نام‌های خوانا تبدیل می‌کنیم
# df_clean['icu_status'] = df_clean['icu'].map({1: 'ICU', 2: 'Non-ICU'})

# # استخراج متغیر مورد نظر برای نمونه‌گیری
# population_age = df_clean['age'].dropna().values
df = df.drop(columns=['id'],axis=1)
df.head()
df.shape

(566602, 22)

### Nan-values

In [3]:
for col in df.columns:
    unique_vals = df[col].unique()
    if len(unique_vals) <= 5:
        print(f"{col}: {unique_vals}")


sex: ['male' 'female']
patient_type: ['hospitalized' 'not hospitalized']
intubed: [nan 'No' 'Yes']
pneumonia: ['No' 'Yes' nan]
pregnancy: [nan 'No' 'Yes']
diabetes: ['No' 'Yes' nan]
copd: ['No' 'Yes' nan]
asthma: ['No' 'Yes' nan]
inmsupr: ['No' 'Yes' nan]
hypertension: ['No' 'Yes' nan]
other_disease: ['No' 'Yes' nan]
cardiovascular: ['No' 'Yes' nan]
obesity: ['No' 'Yes' nan]
renal_chronic: ['No' 'Yes' nan]
tobacco: ['No' 'Yes' nan]
contact_other_covid: ['No' nan 'Yes']
covid_res: ['Positive' 'Negative' 'Not Yet Determined']
icu: [nan 'Not Admitted' 'Admitted']


In [4]:
df.loc[df['sex'] == 'male', 'pregnancy'] = "No"
nan_num = df[df['pregnancy'].isna()].shape[0]
df['pregnancy'] = df['pregnancy'].fillna('No') 
df['contact_other_covid'] = df['contact_other_covid'].fillna('Unknown')

cols = [
    'diabetes','copd','asthma','inmsupr','hypertension',
    'other_disease','cardiovascular','obesity','pneumonia',
    'renal_chronic','tobacco'
]

df[cols] = df[cols].fillna('No')

df['died'] = df['date_died'].copy().notna().astype(int)
df['date_died'] = df['date_died'].fillna(0)


# هر کسی که ICU Admitted بوده، قطعاً بستری بوده است
df.loc[df['icu'] == 'Admitted', 'patient_type'] = 'hospitalized'
df.loc[df['intubed'] == 'Yes', 'patient_type'] = 'hospitalized'

# Outpatients: definitely not intubed or in ICU
df.loc[df['patient_type'] == 'not hospitalized', 'intubed'] = 'No'
df.loc[df['patient_type'] == 'not hospitalized', 'icu'] = 'No'
df[df['intubed'].isna()].shape[0]
df['icu'] = df['icu'].replace('Admitted', 'Yes')

# Inpatients: treat remaining NaN as "No" (most will not be intubed/ICU)
df['intubed'] = df['intubed'].fillna('No')
df['icu']     = df['icu'].fillna('No')




In [5]:
for col in df.columns:
    unique_vals = df[col].unique()
    if len(unique_vals) <= 5:
        print(f"{col}: {unique_vals}")


sex: ['male' 'female']
patient_type: ['hospitalized' 'not hospitalized']
intubed: ['No' 'Yes']
pneumonia: ['No' 'Yes']
pregnancy: ['No' 'Yes']
diabetes: ['No' 'Yes']
copd: ['No' 'Yes']
asthma: ['No' 'Yes']
inmsupr: ['No' 'Yes']
hypertension: ['No' 'Yes']
other_disease: ['No' 'Yes']
cardiovascular: ['No' 'Yes']
obesity: ['No' 'Yes']
renal_chronic: ['No' 'Yes']
tobacco: ['No' 'Yes']
contact_other_covid: ['No' 'Unknown' 'Yes']
covid_res: ['Positive' 'Negative' 'Not Yet Determined']
icu: ['No' 'Yes' 'Not Admitted']
died: [0 1]


In [6]:
nan_nums = df.isna().sum()
nan_percentage = (nan_nums/df.shape[0]*100).round(2)
pd.concat(
    [nan_nums, nan_percentage],
    axis=1,
    keys=['nan_numbers', 'nan_percentage']
)

,nan_numbers,nan_percentage
sex,0,0.0
patient_type,0,0.0
entry_date,0,0.0
date_symptoms,0,0.0
date_died,0,0.0
intubed,0,0.0
pneumonia,0,0.0
age,0,0.0
pregnancy,0,0.0
diabetes,0,0.0


### Encoding categorical data

In [7]:
from sklearn.preprocessing import LabelEncoder

df['sex'] = df['sex'].map({
    'male': 2,
    'female': 1
})

df['pregnancy'] = df['pregnancy'].map({
    'Yes': 1,
    'No': 0
})

# encoding diseases
cols = ['diabetes',
'copd',
'asthma',
'hypertension',
'cardiovascular',
'obesity',
'renal_chronic',
'inmsupr',
'other_disease',
'tobacco',]

for col in cols:
    le = LabelEncoder()
    
    non_null = df[col].dropna()          # only real values
    df.loc[non_null.index, col] = le.fit_transform(non_null)



cols = ['contact_other_covid',
'covid_res',]

for col in cols:
    le = LabelEncoder()
    
    non_null = df[col].dropna()          # only real values
    df.loc[non_null.index, col] = le.fit_transform(non_null)
    
    
    
# encoding clinical status
cols = [
'pneumonia',
'intubed',
'icu',
'patient_type',]

for col in cols:
    le = LabelEncoder()
    non_null = df[col].dropna()          # only real values
    df.loc[non_null.index, col] = le.fit_transform(non_null)
    

### Encoding date-time data
df['date_symptoms'] = pd.to_datetime(df['date_symptoms'], dayfirst=True)
df['entry_date'] = pd.to_datetime(df['entry_date'], dayfirst=True)
df['date_died'] = pd.to_datetime(df['date_died'], dayfirst=True)
df['timestamp_entry_date'] = df['entry_date'].astype('int64') // 10**9 
df['timestamp_date_symptoms'] = df['date_symptoms'].astype('int64') // 10**9 

In [8]:
for col in df.columns:
    unique_vals = df[col].unique()
    if len(unique_vals) <= 5:
        print(f"{col}: {unique_vals}")


sex: [2 1]
patient_type: [0 1]
intubed: [0 1]
pneumonia: [0 1]
pregnancy: [0 1]
diabetes: [0 1]
copd: [0 1]
asthma: [0 1]
inmsupr: [0 1]
hypertension: [0 1]
other_disease: [0 1]
cardiovascular: [0 1]
obesity: [0 1]
renal_chronic: [0 1]
tobacco: [0 1]
contact_other_covid: [0 1 2]
covid_res: [2 0 1]
icu: [0 2 1]
died: [0 1]


In [9]:
df.describe()

,sex,entry_date,date_symptoms,date_died,age,pregnancy,died,timestamp_entry_date,timestamp_date_symptoms
count,566602.000000,566602,566602,566602,566602.000000,566602.000000,566602.000000,5.666020e+05,5.666020e+05
mean,1.506726,2020-05-25 16:38:01.576132608,2020-05-22 00:39:34.695464960,1973-03-21 05:21:42.692189600,42.622483,0.007171,0.063847,1.590425e+09,1.590108e+09
min,1.000000,2020-01-01 00:00:00,2020-01-01 00:00:00,1970-01-01 00:00:00,0.000000,0.000000,0.000000,1.577837e+09,1.577837e+09
25%,1.000000,2020-05-11 00:00:00,2020-05-07 00:00:00,1970-01-01 00:00:00,31.000000,0.000000,0.000000,1.589155e+09,1.588810e+09
50%,2.000000,2020-05-31 00:00:00,2020-05-27 00:00:00,1970-01-01 00:00:00,41.000000,0.000000,0.000000,1.590883e+09,1.590538e+09
75%,2.000000,2020-06-15 00:00:00,2020-06-11 00:00:00,1970-01-01 00:00:00,53.000000,0.000000,0.000000,1.592179e+09,1.591834e+09
max,2.000000,2020-06-29 00:00:00,2020-06-29 00:00:00,2020-06-29 00:00:00,120.000000,1.000000,1.000000,1.593389e+09,1.593389e+09
std,0.499955,NaN,NaN,NaN,16.659973,0.084377,0.244481,2.156197e+06,2.163914e+06


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566602 entries, 0 to 566601
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   sex                      566602 non-null  int64         
 1   patient_type             566602 non-null  object        
 2   entry_date               566602 non-null  datetime64[ns]
 3   date_symptoms            566602 non-null  datetime64[ns]
 4   date_died                566602 non-null  datetime64[ns]
 5   intubed                  566602 non-null  object        
 6   pneumonia                566602 non-null  object        
 7   age                      566602 non-null  int64         
 8   pregnancy                566602 non-null  int64         
 9   diabetes                 566602 non-null  object        
 10  copd                     566602 non-null  object        
 11  asthma                   566602 non-null  object        
 12  inmsupr         

In [11]:
cat_cols = [
    'patient_type','intubed','pneumonia','diabetes',
    'copd','asthma','inmsupr','hypertension',
    'other_disease','cardiovascular','obesity',
    'renal_chronic','tobacco','contact_other_covid',
    'covid_res','icu','died'
]

df[cat_cols] = df[cat_cols].astype('int8')

In [12]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566602 entries, 0 to 566601
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   sex                      566602 non-null  int64         
 1   patient_type             566602 non-null  int8          
 2   entry_date               566602 non-null  datetime64[ns]
 3   date_symptoms            566602 non-null  datetime64[ns]
 4   date_died                566602 non-null  datetime64[ns]
 5   intubed                  566602 non-null  int8          
 6   pneumonia                566602 non-null  int8          
 7   age                      566602 non-null  int64         
 8   pregnancy                566602 non-null  int64         
 9   diabetes                 566602 non-null  int8          
 10  copd                     566602 non-null  int8          
 11  asthma                   566602 non-null  int8          
 12  inmsupr         